In [23]:
import pandas as pd
import copy

import DiversityMetric
import functions

In [24]:
import importlib
importlib.reload(DiversityMetric)
importlib.reload(functions)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sanne\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


<module 'functions' from 'C:\\Users\\sanne\\PycharmProjects\\RADio-\\functions.py'>

In [25]:
sample_size = 20000

In [26]:
articles = functions.process_articles('data/MINDsmall_dev/news.tsv')
behaviors = functions.process_behavior('data/MINDsmall_dev/behaviors.tsv', sample_size)

In [27]:
lstur = functions.process_predictions('data/MINDsmall_dev/lstur_pred_small.json', 'lstur')
nrms = functions.process_predictions('data/MINDsmall_dev/nrms_pred_small.json', 'nrms')
most_popular = functions.process_predictions('data/MINDsmall_dev/pop_pred_small.json', 'most_popular')
random = functions.process_random(copy.deepcopy(lstur))

predictions = pd.concat([lstur, nrms, most_popular, random]).reset_index()

# Configuring the Diversity metrics

## Calibration

Calibration calculates to which extent a recommendation is tailored to a user's preferences. In this initialization, we compare the recommendation to what a user has consumed in the past. 

**Feature**: article category <br>
**Context**: User history <br>
**Feature type**: categorical, here single but could be multi <br>
**Rank-aware**: both recommendation and context <br>
**Desired value**: Low divergence if you want to tailor to a user's preferences; higher if you want to help them encounter new things


In [28]:
Calibration = DiversityMetric.Metric(
    feature_type='cat', 
    rank_aware_recommendation=True,
    rank_aware_context=True,
    divergence='JSD',
    context = 'dynamic'
    )

def calculate_calibration(recommendation, impression_index):
    recommendation_features = functions.select(recommendation, 'category')
    context_features = functions.select(behaviors.loc[impression_index]['history'], 'category')
    if context_features and recommendation_features:
        return Calibration.compute(recommendation_features, context_features)
    return None
    

## Fragmentation

Fragmentation calculates to what extent users that received recommendation have a shared understanding, or: the amount of overlap. Ideally, we would want individual articles to be clustered into stories, as users can read slightly different articles about the same topic but still have a shared understanding. However, as this information is not present in this dataset, we use article subcategory as an approximation. See also "Improving and Evaluating the Detection of Fragmentation in News Recommendations with the Clustering of News Story Chains" by Polimeno et al. for inspiration on how story clustering could work. 


Fragmentation is conceptually related to the filter bubble. When there is low Fragmentation, users see similar topics and thus have a shared understanding of the system. On the other hand, with high Fragmentation what people receive does not overlap: they exist in a personal 'bubble'. This can still be desirable, for example when the goal is to help users specialize in their domain of choice. <br>
As we need to compare recommendations to other users' recommendations, Fragmentation can be computationally heavy. Its performance is highly dependent on the number of other users we compare to. 

**Feature**: article category <br>
**Context**: Other users' recommendations <br>
**Feature type**: categorical, single <br>
**Rank-aware**: both recommendation and context <br>
**Desired value**: low when we want a shared understanding, high when we want specialization

In [29]:
Fragmentation = DiversityMetric.Metric(
    feature_type='cat', 
    rank_aware_recommendation=True,
    rank_aware_context=True,
    divergence='JSD',
    context = 'dynamic'
    )

def calculate_fragmentation(recommendation, user_sample):   
    recommendation_features = functions.select(recommendation, 'subcategory')
    # user_sample = lstur.sample(5).recommendation
    score = 0
    for recommendation_to_other_user in user_sample:
        context_features = functions.select(recommendation_to_other_user, 'subcategory')
        score += Fragmentation.compute(recommendation_features, context_features)
    # average over the size of our sample 
    average = score/len(user_sample)
    return average

## Activation

Activation does...

**Feature**: Absolute sentiment scores <br>
**Context**: All articles published <br>
**Feature type**: continuous <br>
**Rank-aware**: Only the recommendation <br>
**Desired value**: Up to interpretation, and dependent on a) whether the general tone of articles published is emotional or neutral, and b) whether we want this recommendation to reflect that tone or not.



In [30]:
Activation = DiversityMetric.Metric(
    feature_type='cont', 
    rank_aware_recommendation=True,
    rank_aware_context=False,
    divergence='JSD',
    context = 'static'
    )

# This variable helps the metrics to be calculated more efficiently, as we don't need to retrieve the sentiment scores of all articles every time.
all_article_sentiments = list(articles['absolute_sentiment_score'])

def calculate_activation(recommendation):   
    recommendation_features = functions.select(recommendation, 'absolute_sentiment_score')
    context_features = all_article_sentiments
    if context_features and recommendation_features:
        activation =  Activation.compute(recommendation_features, context_features)
        return activation
    return None

## Representation

Representation does...

**Feature**: People mentioned <br>
**Context**: All articles published <br>
**Feature type**: categorical, multi <br>
**Rank-aware**: Only the recommendation <br>
**Desired value**: Low if we want the recommendation to reflect the chosen context distribution

In [31]:
Representation = DiversityMetric.Metric(
    feature_type='cat_m', 
    discount_recommendation=True,
    discount_context=False,
    divergence='JSD',
    context = 'static'
    )

# Only apply Representation to hard news
news = articles[articles['category'] == 'news']
# This variable helps the metrics to be calculated more efficiently, as we don't need to retrieve the sentiment scores of all articles every time.
all_news_persons = list(news[news['persons'].map(len) > 0].persons)

def calculate_representation(recommendation): 
    is_news = news.index.intersection(recommendation)
    recommendation_features = functions.select(is_news, 'persons')
    context_features = all_news_persons
    if context_features and recommendation_features:
        representation =  Representation.compute(recommendation_features, context_features)
        return representation
    return None

## Alternative Voices

Alternative Voices does...

**Feature**: People mentioned <br>
**Context**: The user's history <br>
**Feature type**: categorical, multi <br>
**Rank-aware**: Both the recommendation and the context<br>
**Desired value**: Higher divergence if we want the user to encounter new perspectives; lower if we want them to find their niche.

In [32]:
AlternativeVoices = DiversityMetric.Metric(
    # we currently don't have this information
    feature_type='cat_m', 
    discount_recommendation=True,
    discount_context=True,
    divergence='JSD',
    context = 'dynamic'
    )

def calculate_alternative_voices(recommendation, impression_index): 
    recommendation_features = functions.select(recommendation, 'persons')
    context_features = functions.select(behaviors.loc[impression_index]['history'], 'persons')
    if context_features and recommendation_features:
        alternative_voices =  AlternativeVoices.compute(recommendation_features, context_features)
        return alternative_voices
    return None

# Calculating the metrics

In [33]:
def metrics(row):
    recommendation = row['recommendation']
    impression_index = row['impr_index']
    
    calibration = calculate_calibration(recommendation, impression_index)
    
    user_sample = predictions[predictions['algorithm'] == row['algorithm']].sample(5).recommendation
    fragmentation = calculate_fragmentation(recommendation, user_sample)
    
    activation = calculate_activation(recommendation)    
        
    representation = calculate_representation(recommendation)
    
    alternative_voices = calculate_alternative_voices(recommendation, impression_index)   

    return calibration, fragmentation, activation, representation, alternative_voices




In [ ]:
predictions['metrics'] = predictions.apply(metrics, axis=1)
predictions['calibration'], predictions['fragmentation'], predictions['activation'], predictions['representation'], predictions['alternative_voices'] = zip(*predictions.metrics)

# Visualizing results

In [ ]:
m = ['calibration', 'fragmentation', 'activation', 'representation', 'alternative_voices']

print("Mean")
print(predictions.groupby('algorithm')[m].mean())
print(" ")
print("Standard deviation")
print(predictions.groupby('algorithm')[m].std())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



fig, axs = plt.subplots(ncols=len(m))
fig.set_size_inches(16, 4)
for a, metric in enumerate(m):
    sns.violinplot(data=predictions, x=metric, hue='algorithm', inner="quart", split=False, ax=axs[a])
